# HW2 §3 — Colab runner

**Set runtime to A100** (Runtime → Change runtime type → A100 GPU).  
Results save to `artifacts/colab_results/`. Run the last cell to download `colab_results.zip`.

In [ ]:
# Install — run once
!pip install uv -q
!uv sync
!pip install vllm -q

In [ ]:
import json, sys, time, zipfile, torch
from pathlib import Path

assert torch.cuda.is_available(), "Switch to GPU runtime!"
print(f"GPU: {torch.cuda.get_device_name(0)}  "
      f"({torch.cuda.get_device_properties(0).total_memory/1024**3:.0f} GB)")

DEVICE      = "cuda"
MODEL_NAME  = "Qwen/Qwen2.5-Math-1.5B"
EVAL_N      = 256          # examples per baseline split
GRPO_STEPS  = 50
K_SC        = 5            # self-consistency samples

REPO        = Path(".")
ARTIFACTS   = REPO / "artifacts" / "colab_results"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO))
from alignment.eval import (
    load_gsm8k_examples, build_prompts, _extract_ground_truth,
    evaluate_vllm, write_evaluation_results,
    _load_vllm_model, _make_sampling_params,
)
from alignment.grpo import train_grpo
from alignment.prompts import COT_PROMPT_TEMPLATE, DIRECT_PROMPT_TEMPLATE
from alignment.rewards import answer_tag_reward_fn

In [ ]:
# Load dataset splits once
test_examples  = load_gsm8k_examples("test")[:EVAL_N]
train_examples = load_gsm8k_examples("train")
val_examples   = load_gsm8k_examples("test")          # full test set for GRPO validation
ground_truths  = _extract_ground_truth(test_examples)
print(f"Eval split: {len(test_examples)} examples  |  Train: {len(train_examples)}")

In [ ]:
# Load vLLM model (used for all three baselines, then freed before GRPO)
t0 = time.time()
llm = _load_vllm_model(MODEL_NAME)
print(f"vLLM model loaded in {time.time()-t0:.1f}s")

## §3.1 — Direct prediction baseline

In [ ]:
prompts_direct = build_prompts(test_examples, DIRECT_PROMPT_TEMPLATE)
sp_direct      = _make_sampling_params(max_tokens=512, temperature=1.0, n=1)

t0 = time.time()
r31 = evaluate_vllm(llm, answer_tag_reward_fn, prompts_direct, sp_direct,
                    ground_truths=ground_truths, num_return_sequences=1)
r31["elapsed_seconds"] = time.time() - t0

print(f"n={r31['n']}  format={r31['mean_format_reward']:.3f}  "
      f"answer={r31['mean_answer_reward']:.3f}  ({r31['elapsed_seconds']/60:.1f} min)")
write_evaluation_results(r31, ARTIFACTS / "direct_baseline.json")

## §3.2 — Chain-of-Thought baseline

In [ ]:
prompts_cot = build_prompts(test_examples, COT_PROMPT_TEMPLATE)
sp_cot      = _make_sampling_params(max_tokens=1024, temperature=1.0, n=1)

t0 = time.time()
r_cot = evaluate_vllm(llm, answer_tag_reward_fn, prompts_cot, sp_cot,
                      ground_truths=ground_truths, num_return_sequences=1)
r_cot["elapsed_seconds"] = time.time() - t0

print(f"n={r_cot['n']}  format={r_cot['mean_format_reward']:.3f}  "
      f"answer={r_cot['mean_answer_reward']:.3f}  ({r_cot['elapsed_seconds']/60:.1f} min)")
write_evaluation_results(r_cot, ARTIFACTS / "cot_baseline.json")

## §3.2 — Self-consistency (K=5, CoT prompt, majority vote)

In [ ]:
sp_sc = _make_sampling_params(max_tokens=1024, temperature=1.0, n=K_SC)

t0 = time.time()
r_sc = evaluate_vllm(llm, answer_tag_reward_fn, prompts_cot, sp_sc,
                     ground_truths=ground_truths, num_return_sequences=K_SC)
r_sc["k"] = K_SC
r_sc["elapsed_seconds"] = time.time() - t0

print(f"n={r_sc['n']}  K={K_SC}  format={r_sc['mean_format_reward']:.3f}  "
      f"answer={r_sc['mean_answer_reward']:.3f}  ({r_sc['elapsed_seconds']/60:.1f} min)")
write_evaluation_results(r_sc, ARTIFACTS / f"self_consistency_k{K_SC}.json")

In [ ]:
# Free vLLM GPU memory before loading training model
del llm
import gc; gc.collect()
torch.cuda.empty_cache()
print("vLLM freed.")

## §3.5 — GRPO training

Runs twice — with and without std normalization (~45 min each on A100).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer_g = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer_g.pad_token_id is None:
    tokenizer_g.pad_token_id = tokenizer_g.eos_token_id

policy_std = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")
policy_std.train()

t0 = time.time()
hist_std = train_grpo(
    policy_model=policy_std, tokenizer=tokenizer_g,
    train_examples=train_examples, val_examples=val_examples,
    reward_fn=answer_tag_reward_fn,
    prompt_template=str(COT_PROMPT_TEMPLATE),
    n_grpo_steps=GRPO_STEPS, learning_rate=1e-5, advantage_eps=1e-6,
    rollout_batch_size=32, group_size=8, sampling_temperature=1.0,
    sampling_min_tokens=4, sampling_max_tokens=256,
    epochs_per_rollout_batch=1, train_batch_size=32,
    gradient_accumulation_steps=16, cliprange=1.0,
    normalize_by_std=True, log_every=1, val_every=5, val_size=256, device=DEVICE,
)
hist_std["elapsed_seconds"] = time.time() - t0
print(f"Done in {hist_std['elapsed_seconds']/60:.1f} min")

with open(ARTIFACTS / "grpo_history_std.json", "w") as f:
    json.dump(hist_std, f, indent=2)

del policy_std; gc.collect(); torch.cuda.empty_cache()

In [ ]:
policy_nostd = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")
policy_nostd.train()

t0 = time.time()
hist_nostd = train_grpo(
    policy_model=policy_nostd, tokenizer=tokenizer_g,
    train_examples=train_examples, val_examples=val_examples,
    reward_fn=answer_tag_reward_fn,
    prompt_template=str(COT_PROMPT_TEMPLATE),
    n_grpo_steps=GRPO_STEPS, learning_rate=1e-5, advantage_eps=1e-6,
    rollout_batch_size=32, group_size=8, sampling_temperature=1.0,
    sampling_min_tokens=4, sampling_max_tokens=256,
    epochs_per_rollout_batch=1, train_batch_size=32,
    gradient_accumulation_steps=16, cliprange=1.0,
    normalize_by_std=False, log_every=1, val_every=5, val_size=256, device=DEVICE,
)
hist_nostd["elapsed_seconds"] = time.time() - t0
print(f"Done in {hist_nostd['elapsed_seconds']/60:.1f} min")

with open(ARTIFACTS / "grpo_history_nostd.json", "w") as f:
    json.dump(hist_nostd, f, indent=2)

del policy_nostd; gc.collect(); torch.cuda.empty_cache()

## Plot + download

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for hist, label in [(hist_std, "with std norm"), (hist_nostd, "without std norm")]:
    val_steps = list(range(5, GRPO_STEPS + 1, 5))[:len(hist["val_reward"])]
    ax.plot(val_steps, hist["val_reward"], marker="o", label=label)
ax.set_xlabel("GRPO step"); ax.set_ylabel("Val answer reward")
ax.set_title("GRPO validation reward"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ARTIFACTS / "grpo_training_curves.png", dpi=150)
plt.show()

In [ ]:
zip_path = Path("colab_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in ARTIFACTS.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(REPO))
print(f"Zipped: {zip_path}  ({zip_path.stat().st_size/1024**2:.1f} MB)")

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Download from the file browser on the left.")